# Advanced Toxicity Prediction Model Analyses

This notebook runs advanced evaluations on the `tox-ames` model predictions, covering:
1. **Consensus Ensemble Modeling:** Combining Chemprop MPNN, ChemBERTa, and Fingerprint MLP predictions.
2. **Error Profiling by Physicochemical Properties:** Analysis of accuracy, FPR, and FNR across molecular weight (MW), LogP, and Fsp3 fractions.
3. **Uncertainty Quantification (UQ):** Accuracy vs. prediction confidence threshold curves.
4. **Applicability Domain Mapping:** Testing model accuracy as a function of the chemical similarity (Tanimoto distance) to the training set.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import IPython.display as ipydisplay
import matplotlib.pyplot as plt
from tox_ames.utils.advanced_analyses import (
    build_consensus_ensemble,
    calculate_metrics,
    profile_errors_by_property,
    plot_confidence_accuracy_curve,
    get_train_test_split_smiles,
    compute_similarity_to_training,
    collect_metrics_by_model,
    plot_cv_metrics_comparison,
)
import scienceplots

plt.style.use("science")

BASE_DIR = Path("logs/combined")
OUTPUT_DIR = BASE_DIR / "advanced_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = Path("Figs/advanced_analysis")
FIG_DIR.mkdir(parents=True, exist_ok=True)


## 1. Consensus Ensemble Modeling
We load predictions from our best model configurations (MPNN, ChemBERTa, and Fingerprint MLP) and compute an averaged probability ensemble prediction.

In [ ]:
model_names = [
    "exp-mpnn_mpnn_mean-scf",
    "exp-chemberta_77M-MTR_mean-scf",
    "exp-fp_maccs-scf",
    # "exp-unimolv2_ft-scf",
]
model_dirs = [BASE_DIR / name for name in model_names]

ensemble_df = build_consensus_ensemble(model_dirs)
ensemble_df.to_csv(OUTPUT_DIR / "ensemble_predictions.csv", index=False)

# Calculate cross-validation metrics comparison
records = []

# Ensemble metrics
ens_fold_metrics = calculate_metrics(ensemble_df)
for col in ["Accuracy", "F1", "AUROC", "MCC"]:
    records.append(
        {
            "Model": "Consensus Ensemble",
            "Metric": col,
            "Mean": ens_fold_metrics[col].mean(),
            "Std": ens_fold_metrics[col].std(),
        }
    )

# Individual model metrics
for idx, (name, exp_dir) in enumerate(zip(model_names, model_dirs)):
    pred_df = pd.read_csv(exp_dir / "cv_predictions.csv")
    fold_metrics = calculate_metrics(pred_df)
    for col in ["Accuracy", "F1", "AUROC", "MCC"]:
        records.append(
            {
                "Model": name.split("_")[0].replace("exp-", "").upper(),
                "Metric": col,
                "Mean": fold_metrics[col].mean(),
                "Std": fold_metrics[col].std(),
            }
        )

summary_df = pd.DataFrame(records)
summary_pivot = summary_df.pivot(
    index="Model", columns="Metric", values=["Mean", "Std"]
)
summary_pivot


## 2. Error Profiling by Physicochemical Properties
We calculate continuous properties (Molecular Weight, LogP, and Fsp3) and bin predictions to see where the models display systematic errors (higher FPR or FNR).

In [ ]:
# Bin by Molecular Weight (MW)
mw_profile = profile_errors_by_property(
    ensemble_df, "MW", num_bins=5, save_path=FIG_DIR / "error_profile_MW.png"
)
mw_profile


In [ ]:
# Bin by LogP
logp_profile = profile_errors_by_property(
    ensemble_df, "LogP", num_bins=5, save_path=FIG_DIR / "error_profile_LogP.png"
)
logp_profile


In [ ]:
# Bin by fsp3
fsp3_profile = profile_errors_by_property(
    ensemble_df, "Fsp3", num_bins=5, save_path=FIG_DIR / "error_profile_Fsp3.png"
)
logp_profile


In [ ]:
profiles = profile_errors_by_property(
    ensemble_df,
    ["MW", "LogP"],
    num_bins=5,
    save_path=Path("Figs/error_profile_MW_LogP.png"),
)


In [ ]:
# Manuscript Figure 7 profiles the best individual pipeline (Uni-Mol v2);
# the ensemble counterpart above is reported in the Supporting Information.
unimol_df = pd.read_csv(BASE_DIR / "exp-unimolv2_ft-scf" / "cv_predictions.csv")

unimol_profiles = profile_errors_by_property(
    unimol_df,
    ["MW", "LogP"],
    num_bins=5,
    save_path=Path("Figs/error_profile_MW_LogP_unimolv2.png"),
)


## 3. Uncertainty Quantification (Confidence-Accuracy Curves)
We evaluate if higher confidence predictions (greater distance from the decision boundary) correspond to higher accuracy on the retained subset.

In [ ]:
uq_df = plot_confidence_accuracy_curve(
    ensemble_df, save_path=FIG_DIR / "confidence_accuracy_curve.png"
)
uq_df.head(10)


## 5. Repeated-Holdout Metric Distributions

We visualize metric distributions across the 25 repeated scaffold-balanced holdouts. Because test subsets overlap, these summaries are descriptive and are not treated as independent observations for inferential testing.

In [ ]:
# Plot descriptive metric distributions across repeated holdouts.
# No inferential test is applied because test subsets overlap.
metrics_by_model = collect_metrics_by_model(ensemble_df, model_dirs, model_names)
plot_cv_metrics_comparison(
    metrics_by_model, save_path=FIG_DIR / "cv_metrics_boxplot.png"
)
